# DQL: SELECT

`SELECT` chooses the columns and expressions that appear in the final result. It answers the question: which fields should be shown to the user or passed to the next processing step?

In PySpark, selecting fewer columns is also a good habit for performance and readability. A wide table may contain many fields, but most analysis only needs a small set of columns. Selecting only the needed columns makes the output easier to inspect and can reduce work when Spark reads columnar formats such as Parquet.


## CSV Data Source

CSV stores data as plain text rows. The loader creates the table names used by the examples: `emp`, `dept`, `salgrade`, `projects`, and `emp_projects`.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or the current folder.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Parquet Data Source

Parquet stores data in a compressed columnar format. The same table names are created, so the DQL examples work the same way after loading Parquet.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-parquet-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "dept.parquet").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find dept.parquet. Put the Parquet files in ./data or the current folder.")

emp_paths = sorted(DATA_DIR.glob("emp_part_*.parquet"))
if not emp_paths:
    raise FileNotFoundError("Could not find emp_part_*.parquet files in the Parquet data folder.")

print(f"Reading Parquet files from: {DATA_DIR}")

emp = spark.read.parquet(*[str(path) for path in emp_paths])
dept = spark.read.parquet(str(DATA_DIR / "dept.parquet"))
salgrade = spark.read.parquet(str(DATA_DIR / "salgrade.parquet"))
projects = spark.read.parquet(str(DATA_DIR / "projects.parquet"))
emp_projects = spark.read.parquet(str(DATA_DIR / "emp_projects.parquet"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## SELECT

Use `select` to choose columns. In PySpark you can use the DataFrame API or Spark SQL.

In plain English: `SELECT` means "show me these columns." It does not remove rows by itself; it controls which columns appear in the final output and the order in which those columns are displayed.

Common uses:

* Choose a subset of columns from a large table.
* Reorder columns so the most important fields appear first.
* Create calculated columns using expressions such as `sal * 12`.
* Rename columns with aliases so output names are easier to read.

Key points:

* `emp.select(...)` is the PySpark DataFrame version.
* `SELECT ... FROM emp` is the SQL version.
* Both examples return employee columns from the same `emp` data.
* `SELECT` is often one of the final shaping steps in a query.

Watch out: selecting a column does not filter rows. To remove rows, use `WHERE` or `filter`.


In [ ]:
emp.select("empno", "ename", "job", "sal", "deptno").show(10)

spark.sql("""
SELECT empno, ename, job, sal, deptno
FROM emp
LIMIT 10
""").show()
